# Data Sharding

This tutorial shows how to use `JaxLoader` and `EQXTrainer` with **multi-device data parallelism**.

We use `jax.config.update("jax_num_cpu_devices", N)` to simulate multiple devices locally — useful for testing sharding logic without GPUs/TPUs.

> **Note:** `jax_num_cpu_devices` must be set *before* any JAX operation is performed (ideally at the very top of your script/notebook).

In [ ]:
import jax
jax.config.update("jax_num_cpu_devices", 4)  # simulate 4 CPU devices

import jax.numpy as jnp
import jax.sharding as jsd
import numpy as np
import optax
import equinox as eqx
from jax.experimental import mesh_utils

from trainax import EQXTrainer, JaxLoader, StepOutput, ValStepOutput, EpochLogger

print("Available devices:", jax.devices())

## 1. Data

The batch size must be divisible by the number of shards. Here we use 4 devices, so batch size must be a multiple of 4.

In [ ]:
np.random.seed(0)
n_devices = len(jax.devices())
batch_size = 32  # 32 / 4 = 8 samples per device

x = np.random.randn(800, 16).astype(np.float32)
y = (x @ np.random.randn(16, 1)).astype(np.float32).squeeze(-1)

trainloader = JaxLoader({"x": x[:600], "y": y[:600]}, batch_size=batch_size)
valloader   = JaxLoader({"x": x[600:], "y": y[600:]}, batch_size=batch_size)

print(f"Devices: {n_devices}, batch size: {batch_size}, samples/device: {batch_size // n_devices}")

## 2. Understanding sharding in trainax

Pass `data_sharding=N` to the trainer constructor, where `N` is the number of devices to shard data across. The trainer:

1. Creates a `jax.sharding.Mesh` with `N` devices along the `"data"` axis.
2. Calls `loader.set_sharding(sharding)` on both train and val loaders.
3. Each batch is then a `jax.Array` sharded across all devices using `jax.make_array_from_process_local_data`.

You can also configure sharding manually:

In [ ]:
# Manual sharding setup (equivalent to data_sharding=4 in the trainer)
devices = jax.devices()
mesh = jsd.Mesh(mesh_utils.create_device_mesh((n_devices,), devices=devices), ("data",))
sharding = jsd.NamedSharding(mesh, jsd.PartitionSpec("data"))

# Demonstrate what a sharded batch looks like
demo_loader = JaxLoader({"x": x[:64], "y": y[:64]}, batch_size=32)
demo_loader.set_sharding(sharding)

for batch in demo_loader:
    print("Batch x type:", type(batch["x"]))
    print("Batch x shape:", batch["x"].shape)
    print("Sharding:", batch["x"].sharding)
    break

## 3. Model and step functions

Step functions work identically whether data is sharded or not — JAX handles the parallelism transparently.

In [ ]:
model = eqx.nn.MLP(16, 1, width_size=32, depth=2, key=jax.random.key(0))

def train_step(model, batch):
    def loss_fn(m):
        yhat = jax.vmap(m)(batch["x"]).squeeze(-1)
        return jnp.mean((yhat - batch["y"]) ** 2), yhat
    (loss, yhat), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(model)
    return StepOutput(loss=loss, y=batch["y"], yhat=yhat, gradients=grads)

def val_step(model, batch):
    yhat = jax.vmap(model)(batch["x"]).squeeze(-1)
    loss = jnp.mean((yhat - batch["y"]) ** 2)
    return ValStepOutput(loss=loss, y=batch["y"], yhat=yhat)

## 4. Train with `data_sharding=4`

In [ ]:
trainer = EQXTrainer(
    n_epochs=10,
    callbacks=[EpochLogger("logger")],
    val_every=2,
    use_rich=False,
    data_sharding=n_devices,  # <- enable data sharding
)

print("Trainer sharding config:", trainer.sharding)

model, _ = trainer.train(
    model, optax.adam(1e-3), train_step, trainloader, val_step, valloader
)

## 5. Verifying sharding

We can confirm that data batches were actually distributed across devices:

In [ ]:
# After trainer.train(), the loaders have been configured with sharding
# Inspect a batch directly
for batch in trainloader:
    x_batch = batch["x"]
    print("Batch type:", type(x_batch))
    print("Shape:", x_batch.shape)
    if hasattr(x_batch, "sharding"):
        print("Sharding:", x_batch.sharding)
        print("Addressable shards:", [s.data.shape for s in x_batch.addressable_shards])
    break

## Key points

- `data_sharding=N` shards each batch's leading axis across `N` devices.
- The batch size **must be divisible** by the number of shards — the loader asserts this.
- For `SingleBatchJaxLoader`, the dataset size doesn't need to be divisible; the loader drops the remainder with a warning.
- JAX's `pmap`/`vmap`/`jit` pick up the sharding automatically — no changes to step functions needed.
- To change sharding after construction, call `trainer.set_sharding({"data": N, "model": M})` or assign `trainer.sharding = ...`.